In [30]:
import pandas as pd
import numpy as np
import kaggle as kaggle
import psycopg2
from sqlalchemy import create_engine
import _mysql_connector


In [31]:
kaggle.api.authenticate()
# Load the dataset from Kaggle
dataset_path = kaggle.api.dataset_download_files('najir0123/walmart-10k-sales-datasets', path='.', unzip=True)


Dataset URL: https://www.kaggle.com/datasets/najir0123/walmart-10k-sales-datasets


In [32]:
# Display the first few rows of the DataFrame
df = pd.read_csv('Walmart.csv')
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


## Data Exploration

In [33]:
df.shape

(10051, 11)

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  object 
 2   City            10051 non-null  object 
 3   category        10051 non-null  object 
 4   unit_price      10020 non-null  object 
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  object 
 7   time            10051 non-null  object 
 8   payment_method  10051 non-null  object 
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 863.9+ KB


In [35]:
df.describe()

,invoice_id,quantity,rating,profit_margin
count,10051.000000,10020.000000,10051.000000,10051.000000
mean,5025.741220,2.353493,5.825659,0.393791
std,2901.174372,1.602658,1.763991,0.090669
min,1.000000,1.000000,3.000000,0.180000
25%,2513.500000,1.000000,4.000000,0.330000
50%,5026.000000,2.000000,6.000000,0.330000
75%,7538.500000,3.000000,7.000000,0.480000
max,10000.000000,10.000000,10.000000,0.570000


In [36]:
df.dtypes

invoice_id          int64
Branch             object
City               object
category           object
unit_price         object
quantity          float64
date               object
time               object
payment_method     object
rating            float64
profit_margin     float64
dtype: object

In [ ]:
# Checking for duplicated rows
df.duplicated().sum()

51

In [ ]:
# Dropping the duplicated records
df.drop_duplicates(inplace=True)


In [39]:
df.duplicated().sum()

0

In [40]:
df.shape

(10000, 11)

## Data Cleaning

In [41]:
df.isnull().sum()

invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

In [42]:
df.dropna(inplace=True)

In [43]:
df.isnull().sum()

invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

In [44]:
# Remove the '$' sign and convert 'unit_price' to float
df['unit_price'] = df['unit_price'].str.replace('$','').astype(float)

df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


In [45]:
# Changing 'Ewallet' to 'E-wallet'
df['payment_method'] = df['payment_method'].str.replace('Ewallet', 'E-wallet')
df['payment_method'].unique().tolist()

['E-wallet', 'Cash', 'Credit card']

In [46]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,E-wallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,E-wallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,E-wallet,5.3,0.48


In [47]:
df['Total_Price'] = (df['unit_price'] * df['quantity']).round(2)
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,Total_Price
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,E-wallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,E-wallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,E-wallet,5.3,0.48,604.17


In [48]:

df.columns = df.columns.str.lower()
df.head()

,invoice_id,branch,city,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total_price
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,E-wallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,E-wallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,E-wallet,5.3,0.48,604.17


In [ ]:
host = 'localhost'
port = 5432
user = 'postgres'
password = '**********'
database = 'walmart_db'

In [50]:
df.to_csv('walmart_sales_cleaned.csv', index=False)

In [51]:
try:
    # Create a connection string
    conn_string = f"postgresql://{user}:{password}@{host}:{port}/{database}"
    
    # Create a SQLAlchemy engine
    engine = create_engine(conn_string)
    
    # Load the DataFrame into the PostgreSQL database
    df.to_sql('walmart_sales', engine, if_exists='replace', index=False)
    
    print("Data loaded successfully into PostgreSQL database.")
except Exception as e:
    print("Error loading data into PostgreSQL database:", e)
    

Data loaded successfully into PostgreSQL database.
